# LPT source review

Interactive review of **long period radio transients (LPTs)** in OVRO-LWA
time–frequency data. Positions follow Rea et al. ([arXiv:2601.10393](https://arxiv.org/abs/2601.10393),
Table 1): Galactic \((l, b)\) are converted to ICRS RA/Dec before analysis.
The source dropdown lists only targets with **Dec > −20°**.

For each selected source and **heatmap method**:

1. Build a time × frequency map (tracked pixel, patch statistic, patch maximum, or
   Gaussian patch fit — same options as `jupiter_flux_review.ipynb`, plus `mad`, `std`,
   `mean`, `min`).
2. **Click** a cell in the heatmap to load that slice in **astrowidget.SkyWidget**
   centered on the source.

Launch with: `pixi run jupyter lab`


In [ ]:
# Edit before running if your paths or cuts differ.
from pathlib import Path

#ZARR_PATH = Path("/fast/claw/I-10m-Taper-25Jan.zarr")
ZARR_PATH = Path("/fast/claw/pipelineQA-phase2-V-Taper-Robust-0-20241228.zarr/")

MIN_DEC_DEG = -10.0  # dropdown: sources with Dec > this value (degrees)
PATCH_SCALE = 5.0  # patch half-width = ceil(scale * max beam FWHM in pixels)
SKY_FOV_DEG = 8.0

# Default heatmap fill: tracked pixel, patch stats, patch_max, or patch_fit
HEATMAP_METHOD = "patch_max"
PATCH_FIT_MAX_REDUCED_CHI_SQUARED = 10.0


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import math
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import ovro_lwa_portal as ovro
import panel as pn
import param
import xarray as xr
import astropy.units as u
from astropy.coordinates import SkyCoord
from bokeh.events import Tap
from bokeh.models import ColumnDataSource, FixedTicker, HoverTool, LinearColorMapper
from bokeh.palettes import Inferno256, Magma256
from bokeh.plotting import figure
from astrowidget import SkyWidget

from ovro_lwa_portal.accessor import format_radec_sexagesimal
from ovro_lwa_portal.viz.pipeline_qa_app import (
    ACTIVITY_LOG_HEIGHT_PX,
    ScrollLog,
    _format_activity_log_html,
    _patch_astrowidget_get_wcs,
    _push_panel_layout,
    _schedule_ipython_main,
)

_patch_astrowidget_get_wcs()
pn.extension("bokeh", sizing_mode="stretch_width")

HEATMAP_METHOD_OPTIONS = [
    "dynamic_spectrum",
    "patch_max",
    "patch_fit",
    "mad",
    "std",
    "mean",
    "min",
]


In [ ]:
# Galactic (l, b) in degrees and P1 in seconds — Rea et al. 2026, arXiv:2601.10393, Table 1
LPT_CATALOG_GALACTIC = [
    {"name": "GCRT J1745-3009", "l": 358.8911, "b": -0.5409, "period_s": 4620.72},
    {"name": "GLEAM-X J1627-5235", "l": 332.4646, "b": -2.6009, "period_s": 1091.17},
    {"name": "GPM J1839-10", "l": 22.1526, "b": -2.0629, "period_s": 1318.19},
    {"name": "ASKAP J1935+2148", "l": 57.1901, "b": 0.7453, "period_s": 2225.31},
    {"name": "CHIME J0630+25", "l": 187.9709, "b": 7.0606, "period_s": 421.35},
    {"name": "ASKAP/DART J1832-0911", "l": 22.6406, "b": -0.0839, "period_s": 2656.24},
    {"name": "ILT J1101+5521", "l": 150.4551, "b": 55.5199, "period_s": 7531.78},
    {"name": "GLEAM-X J0704-36", "l": 247.8955, "b": -13.6352, "period_s": 10496.55},
    {"name": "ASKAP J1839-0756", "l": 24.5450, "b": -1.0557, "period_s": 23221.70},
    {"name": "ASKAP J1448-6856", "l": 313.1644, "b": -8.4338, "period_s": 5631.07},
    {"name": "CHIME/ILT J1634+44", "l": 70.1692, "b": 42.5754, "period_s": 841.24},
    {"name": "ASKAP J1755-2527", "l": 4.116664, "b": -0.122707, "period_s": 4186.32},
]


def galactic_to_icrs(l_deg: float, b_deg: float) -> SkyCoord:
    """Convert Galactic (l, b) to ICRS equatorial coordinates."""
    return SkyCoord(l=l_deg * u.deg, b=b_deg * u.deg, frame="galactic").icrs


def build_lpt_sources(*, min_dec_deg: float = -20.0) -> list[dict]:
    """LPT catalog entries with ICRS RA/Dec (degrees) north of ``min_dec_deg``."""
    sources: list[dict] = []
    for entry in LPT_CATALOG_GALACTIC:
        icrs = galactic_to_icrs(entry["l"], entry["b"])
        dec_deg = float(icrs.dec.deg)
        if dec_deg <= min_dec_deg:
            continue
        sources.append(
            {
                "name": entry["name"],
                "l": entry["l"],
                "b": entry["b"],
                "period_s": entry["period_s"],
                "ra": float(icrs.ra.deg),
                "dec": dec_deg,
            }
        )
    return sources


def lst_hours_for_dataset(ds: xr.Dataset) -> np.ndarray:
    """Mean local sidereal time (hours) for each dataset time sample."""
    from astropy.coordinates import EarthLocation
    from astropy.time import Time
    from astropy.utils.iers import conf as iers_conf

    observatory = EarthLocation.of_site("ovro")
    mjd = np.asarray(ds.coords["time"].values, dtype=np.float64)
    orig = iers_conf.auto_download
    try:
        iers_conf.auto_download = False
        times = Time(mjd, format="mjd", scale="utc")
        lst_deg = np.asarray(times.sidereal_time("mean", longitude=observatory.lon).deg)
    finally:
        iers_conf.auto_download = orig
    return np.mod(lst_deg / 15.0, 24.0)


def first_valid_sky_slice(dataset: xr.Dataset, freq_idx: int | None = None) -> tuple[int, int]:
    """First time index with finite SKY at the image centre."""
    fi = dataset.sizes["frequency"] // 2 if freq_idx is None else int(freq_idx)
    center = dataset.sizes["l"] // 2
    ts = dataset["SKY"].isel(polarization=0, frequency=fi, l=center, m=center)
    data = ts.data
    vals = np.asarray(data.compute() if hasattr(data, "compute") else data)
    valid = np.flatnonzero(np.isfinite(vals))
    if valid.size == 0:
        raise ValueError("No finite SKY data found at the image center.")
    return int(valid[0]), fi


@dataclass
class HeatmapLoad:
    """Values and optional accessor results for one source/method pair."""

    values: np.ndarray
    patch_fit_result: object | None = None
    patch_stat_result: object | None = None


def compute_lpt_heatmap(
    dataset: xr.Dataset,
    src: dict,
    *,
    method: str,
    scale: float,
    patch_fit_max_reduced_chi_squared: float,
) -> HeatmapLoad:
    """Build the (time, frequency) array used to fill the heatmap."""
    ra = float(src["ra"])
    dec = float(src["dec"])
    if method == "dynamic_spectrum":
        da = dataset.radport.dynamic_spectrum(ra=ra, dec=dec)
        return HeatmapLoad(np.asarray(da.values, dtype=np.float64))
    if method == "patch_fit":
        fit = dataset.radport.patch_fit(
            ra=ra,
            dec=dec,
            scale=scale,
            max_reduced_chi_squared=patch_fit_max_reduced_chi_squared,
            allow_position_offset=True,
        )
        return HeatmapLoad(np.asarray(fit.peak_map.values, dtype=np.float64), patch_fit_result=fit)
    if method == "patch_max":
        result = dataset.radport.patch_statistic(
            ra=ra, dec=dec, statistic="max", scale=scale
        )
        return HeatmapLoad(np.asarray(result.stat_map.values, dtype=np.float64), patch_stat_result=result)
    if method in ("mad", "std", "mean", "min"):
        result = dataset.radport.patch_statistic(
            ra=ra, dec=dec, statistic=method, scale=scale
        )
        return HeatmapLoad(np.asarray(result.stat_map.values, dtype=np.float64), patch_stat_result=result)
    msg = f"Unknown heatmap method {method!r}; expected one of {HEATMAP_METHOD_OPTIONS}"
    raise ValueError(msg)


def _heatmap_index_from_coord(coord: float, n: int) -> int:
    if n <= 0:
        return 0
    return int(np.clip(int(np.floor(float(coord))), 0, n - 1))


def _format_lst_hour_label(lst_hour: float) -> str:
    hour = int(round(float(lst_hour))) % 24
    return f"{hour:02d}h"


def _format_scalar_hover(value: float, *, fmt: str = ".3g") -> str:
    if np.isfinite(value):
        return format(float(value), fmt)
    return "n/a"


def _color_mapper(values: np.ndarray, *, palette=Magma256) -> LinearColorMapper:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return LinearColorMapper(palette=palette, low=0.0, high=1.0, nan_color="#9e9e9e")
    lo, hi = np.percentile(finite, [2, 98])
    if hi <= lo:
        hi = lo + 1.0
    return LinearColorMapper(
        palette=palette,
        low=float(lo),
        high=float(hi),
        nan_color="#9e9e9e",
    )


def _row_hover(arr: np.ndarray) -> list[str]:
    return [_format_scalar_hover(float(v)) for v in arr.ravel()]


def _patch_fit_hover_columns(fit: object) -> dict[str, list[str]]:
    """Pre-formatted Bokeh hover fields for patch-fit (same as Jupiter notebook)."""
    chi2 = np.asarray(fit.reduced_chi_squared_map.values, dtype=np.float64)
    peak = np.asarray(fit.peak_map.values, dtype=np.float64)
    x_off = np.asarray(fit.x_offset_map.values, dtype=np.float64)
    y_off = np.asarray(fit.y_offset_map.values, dtype=np.float64)
    pmax = np.asarray(fit.patch_max_map.values, dtype=np.float64)
    accepted = np.asarray(fit.fit_accepted_map.values, dtype=bool)
    peak_ra, peak_dec = fit.peak_radec_maps()
    ra = np.asarray(peak_ra.values, dtype=np.float64)
    dec = np.asarray(peak_dec.values, dtype=np.float64)

    peak_ra_display: list[str] = []
    peak_dec_display: list[str] = []
    for r, d in zip(ra.ravel(), dec.ravel(), strict=True):
        ra_s, dec_s = format_radec_sexagesimal(float(r), float(d))
        peak_ra_display.append(ra_s)
        peak_dec_display.append(dec_s)

    return {
        "chi2_display": _row_hover(chi2),
        "peak_ra_display": peak_ra_display,
        "peak_dec_display": peak_dec_display,
        "offset_display": [
            (
                f"({x:.2f}, {y:.2f})"
                if np.isfinite(x) and np.isfinite(y)
                else "n/a"
            )
            for x, y in zip(x_off.ravel(), y_off.ravel(), strict=True)
        ],
        "fit_accepted_display": ["yes" if a else "no" for a in accepted.ravel()],
        "patch_max_display": _row_hover(pmax),
        "peak_flux_display": [
            f"{v:.3g} (masked)" if not np.isfinite(v) else f"{v:.3g}"
            for v in peak.ravel()
        ],
    }


def _format_patch_fit_diagnostics(fit: object, time_idx: int, freq_idx: int) -> str:
    diag = fit.cell_diagnostics(time_idx=time_idx, frequency_idx=freq_idx)
    accepted = "yes" if diag["fit_accepted"] else "no (χ² above cut)"
    peak = diag["peak"]
    peak_s = f"{peak:.3g}" if np.isfinite(peak) else "n/a (masked)"
    return (
        f"**patch_fit** t={time_idx} f={freq_idx}: accepted={accepted}, "
        f"χ²_red={diag['reduced_chi_squared']:.3g}, peak={peak_s} Jy, "
        f"peak RA/Dec=({diag['peak_ra']}, {diag['peak_dec']}), "
        f"offset=({diag['x_offset_pixels']:.2f}, {diag['y_offset_pixels']:.2f}) px, "
        f"patch_max={diag['patch_max']:.3g} Jy"
    )


LPT_SOURCES = build_lpt_sources(min_dec_deg=MIN_DEC_DEG)
print(
    f"Catalog: {len(LPT_SOURCES)} LPT sources with Dec > {MIN_DEC_DEG}° "
    f"(of {len(LPT_CATALOG_GALACTIC)} in Rea et al. Table 1)."
)


In [ ]:
class LPTSourceReview(param.Parameterized):
    """LPT heatmap + SkyWidget review (Jupiter-style Panel UI)."""

    select_source = param.Selector(default=None, objects=[], doc="LPT from Rea et al. catalog.")
    heatmap_method = param.Selector(
        default="mad",
        objects=HEATMAP_METHOD_OPTIONS,
        doc="Quantity plotted in the time–frequency heatmap.",
    )
    loading = param.Boolean(default=False)
    status = param.String(default="Opening Zarr store…")
    log_text = param.String(default="")

    def __init__(
        self,
        sources: list[dict],
        zarr_path: Path,
        *,
        patch_scale: float,
        sky_fov_deg: float,
        patch_fit_max_reduced_chi_squared: float,
        **params,
    ) -> None:
        self._sources = sources
        self._source_by_name = {src["name"]: src for src in sources}
        self._zarr_path = Path(zarr_path)
        self._patch_scale = float(patch_scale)
        self._sky_fov_deg = float(sky_fov_deg)
        self._patch_fit_max_chi2 = float(patch_fit_max_reduced_chi_squared)
        self._scroll_log = ScrollLog()
        self._dataset: xr.Dataset | None = None
        self._cache: dict[tuple[str, str], HeatmapLoad] = {}
        self._heatmap_values: np.ndarray | None = None
        self._patch_fit_result: object | None = None
        self._patch_stat_result: object | None = None
        self._current_source: dict | None = None
        self._coord: SkyCoord | None = None
        self._lst_hours: np.ndarray | None = None
        self._freq_mhz: np.ndarray | None = None
        self._sky_widget: SkyWidget | None = None
        self._time_idx = 0
        self._freq_idx = 0
        self._default_time_idx = 0
        self._default_freq_idx = 0

        names = [src["name"] for src in sources]
        default = names[0] if names else None
        super().__init__(select_source=default, **params)
        self.param.select_source.objects = names

        self._heatmap_pane = pn.pane.Bokeh(height=420, sizing_mode="stretch_width")
        self._sky_container = widgets.VBox(
            children=[widgets.HTML("<i>Sky view loads after the Zarr store opens.</i>")],
            layout=widgets.Layout(width="100%", min_height="620px"),
        )
        self._sky_pane = pn.pane.IPyWidget(self._sky_container, height=620, sizing_mode="stretch_width")
        self._status_pane = pn.pane.Markdown("")
        self._log_pane = pn.pane.HTML(
            _format_activity_log_html(""),
            sizing_mode="stretch_width",
            height=ACTIVITY_LOG_HEIGHT_PX,
        )
        self._selector = pn.widgets.Select.from_param(
            self.param.select_source,
            name="LPT source",
            width=360,
        )
        self._method_selector = pn.widgets.Select.from_param(
            self.param.heatmap_method,
            name="Heatmap method",
            width=220,
        )
        self._spinner = pn.indicators.LoadingSpinner(value=False, size=24, name="")
        self._layout = pn.Column(
            pn.Row(
                self._selector,
                self._method_selector,
                self._spinner,
                margin=(0, 0, 8, 0),
            ),
            pn.Column(
                pn.pane.Markdown("**Activity log**"),
                self._log_pane,
                sizing_mode="stretch_width",
            ),
            self._status_pane,
            self._heatmap_pane,
            self._sky_pane,
            sizing_mode="stretch_width",
            max_width=1048,
        )
        self.param.watch(self._on_select_source, "select_source")
        self.param.watch(self._on_heatmap_method_change, "heatmap_method")
        self._log(f"Catalog: {len(sources)} LPT positions (Dec > {MIN_DEC_DEG}°).")
        self._log(f"Zarr: {self._zarr_path}")
        self._open_dataset()

    @property
    def panel(self) -> pn.Column:
        return self._layout

    def _heatmap_method_label(self) -> str:
        labels = {
            "dynamic_spectrum": "tracked centre pixel",
            "patch_max": "patch maximum",
            "patch_fit": "Gaussian patch fit (peak)",
            "mad": "patch MAD",
            "std": "patch std",
            "mean": "patch mean",
            "min": "patch min",
        }
        return labels.get(self.heatmap_method, self.heatmap_method)

    def _set_status(self, text: str) -> None:
        self.status = text
        self._status_pane.object = text

    @param.depends("log_text", watch=True)
    def _sync_log_pane(self) -> None:
        self._log_pane.object = _format_activity_log_html(self.log_text)

    def _sync_log(self) -> None:
        self.log_text = self._scroll_log.text

    def _log(self, message: str) -> None:
        self._scroll_log.append(message)
        self._sync_log()
        _push_panel_layout(self._layout, self._log_pane)

    def _open_dataset(self) -> None:
        self.loading = True
        self._spinner.value = True
        _push_panel_layout(self._layout, self._spinner, self._log_pane)

        def _work() -> None:
            try:
                _schedule_ipython_main(
                    lambda: self._log(f"Opening {self._zarr_path.name} (chunked l,m=512)…")
                )
                ds = ovro.open_dataset(
                    self._zarr_path,
                    chunks={"l": 512, "m": 512},
                )
                t0, f0 = first_valid_sky_slice(ds)
                lst_hours = lst_hours_for_dataset(ds)
                freq_mhz = np.asarray(ds.coords["frequency"].values, dtype=np.float64) / 1e6
            except Exception as exc:
                _schedule_ipython_main(lambda: self._finish_open(None, None, None, None, None, exc))
                return
            _schedule_ipython_main(
                lambda: self._finish_open(ds, t0, f0, lst_hours, freq_mhz, None)
            )

        import threading

        threading.Thread(target=_work, daemon=True).start()

    def _finish_open(
        self,
        ds: xr.Dataset | None,
        default_time_idx: int | None,
        default_freq_idx: int | None,
        lst_hours: np.ndarray | None,
        freq_mhz: np.ndarray | None,
        error: BaseException | None,
    ) -> None:
        if error is not None:
            self.loading = False
            self._spinner.value = False
            self._log(f"ERROR: {error}")
            self._set_status(f"**Load failed:** {error}")
            _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)
            return

        assert ds is not None and lst_hours is not None and freq_mhz is not None
        assert default_time_idx is not None and default_freq_idx is not None

        self._dataset = ds
        self._lst_hours = lst_hours
        self._freq_mhz = freq_mhz
        self._default_time_idx = int(default_time_idx)
        self._default_freq_idx = int(default_freq_idx)
        self._time_idx = self._default_time_idx
        self._freq_idx = self._default_freq_idx

        self._mount_sky_widget(ds)
        self._log(
            f"Opened — {int(ds.sizes['time'])} times × {int(ds.sizes['frequency'])} freqs, "
            f"{int(ds.sizes['l'])}×{int(ds.sizes['m'])} px, WCS={ds.radport.has_wcs}."
        )
        self.loading = False
        self._spinner.value = False
        if self.select_source is not None:
            self._load_heatmap(self.select_source)

    def _on_select_source(self, *_events) -> None:
        name = self.select_source
        if name is None or self._dataset is None or self.loading:
            return
        self._load_heatmap(name)

    def _on_heatmap_method_change(self, *_events) -> None:
        name = self.select_source
        if name is None or self._dataset is None or self.loading:
            return
        self._load_heatmap(name)

    def _load_heatmap(self, name: str) -> None:
        if self._dataset is None:
            return
        src = self._source_by_name[name]
        method = str(self.heatmap_method)
        cache_key = (name, method)
        if cache_key in self._cache:
            self._apply_heatmap(src, self._cache[cache_key])
            return

        self.loading = True
        self._spinner.value = True
        self._log(
            f"Computing {self._heatmap_method_label()} for {name} "
            f"(RA={src['ra']:.4f}°, Dec={src['dec']:.4f}°, scale={self._patch_scale:g})…"
        )
        self._set_status(f"Computing **{self._heatmap_method_label()}** for **{name}**…")
        _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)

        def _work() -> None:
            try:
                payload = compute_lpt_heatmap(
                    self._dataset,
                    src,
                    method=method,
                    scale=self._patch_scale,
                    patch_fit_max_reduced_chi_squared=self._patch_fit_max_chi2,
                )
            except Exception as exc:
                _schedule_ipython_main(lambda: self._finish_heatmap(src, None, exc))
                return
            _schedule_ipython_main(lambda: self._finish_heatmap(src, payload, None))

        import threading

        threading.Thread(target=_work, daemon=True).start()

    def _finish_heatmap(
        self,
        src: dict,
        payload: HeatmapLoad | None,
        error: BaseException | None,
    ) -> None:
        self.loading = False
        self._spinner.value = False
        if error is not None:
            self._log(f"ERROR ({src['name']}): {error}")
            self._set_status(f"**Heatmap failed for {src['name']}:** {error}")
            _push_panel_layout(self._layout, self._status_pane, self._spinner, self._log_pane)
            return

        assert payload is not None
        cache_key = (src["name"], str(self.heatmap_method))
        self._cache[cache_key] = payload
        finite = payload.values[np.isfinite(payload.values)]
        if finite.size:
            self._log(
                f"{src['name']} ({self._heatmap_method_label()}): "
                f"range [{float(finite.min()):.3g}, {float(finite.max()):.3g}]"
            )
        else:
            self._log(f"{src['name']} ({self._heatmap_method_label()}): no finite values")
        self._apply_heatmap(src, payload)

    def _apply_heatmap(self, src: dict, payload: HeatmapLoad) -> None:
        self._current_source = src
        self._heatmap_values = payload.values
        self._patch_fit_result = payload.patch_fit_result
        self._patch_stat_result = payload.patch_stat_result
        self._coord = SkyCoord(ra=src["ra"] * u.deg, dec=src["dec"] * u.deg, frame="icrs")
        self._time_idx, self._freq_idx = self._default_slice(payload.values)

        period_min = src["period_s"] / 60.0
        ra_h = self._coord.ra.to_string(unit=u.hour, precision=1)
        dec_s = self._coord.dec.to_string(unit=u.deg, precision=1)
        self._set_status(
            f"**{src['name']}** — P₁={period_min:.2f} min, "
            f"l={src['l']:.2f}°, b={src['b']:.2f}°, RA={ra_h}, Dec={dec_s} · "
            f"Heatmap: **{self._heatmap_method_label()}** (scale={self._patch_scale:g}) · "
            "**Click the heatmap** to inspect a time/frequency slice."
        )
        self._heatmap_pane.object = self._build_heatmap_figure(payload.values)
        self._update_sky(self._time_idx, self._freq_idx)
        _push_panel_layout(
            self._layout, self._status_pane, self._heatmap_pane, self._sky_pane, self._log_pane
        )

    def _default_slice(self, values: np.ndarray) -> tuple[int, int]:
        finite = np.argwhere(np.isfinite(values))
        if finite.size:
            t_idx, f_idx = finite[len(finite) // 2]
            return int(t_idx), int(f_idx)
        return self._default_time_idx, self._default_freq_idx

    def _mount_sky_widget(self, ds: xr.Dataset) -> None:
        widget = SkyWidget()
        widget.colormap = "inferno"
        widget.background_survey = "DSS"
        widget.stretch = "sqrt"
        widget.invert_horizontal_pan = True
        max_size = max(256, int(ds.sizes["l"]) // 2)
        widget.set_dataset(ds, max_size=max_size)
        widget.update_slice(
            time_idx=self._default_time_idx,
            freq_idx=self._default_freq_idx,
        )
        widget.auto_scale(percentile_low=2, percentile_high=98)
        self._sky_widget = widget
        self._sky_container.children = [widget]

    def _update_sky(self, time_idx: int, freq_idx: int) -> None:
        widget = self._sky_widget
        coord = self._coord
        if widget is None or coord is None:
            return
        widget.update_slice(
            time_idx=int(time_idx),
            freq_idx=int(freq_idx),
            center=coord,
            fov=self._sky_fov_deg * u.deg,
            percentile_low=2,
            percentile_high=98,
        )
        send_state = getattr(widget, "send_state", None)
        if callable(send_state):
            send_state()

    def _on_heatmap_tap(self, time_idx: int, freq_idx: int) -> None:
        self._time_idx = time_idx
        self._freq_idx = freq_idx
        src = self._current_source
        if src is None or self._lst_hours is None or self._freq_mhz is None:
            return

        lst = _format_lst_hour_label(float(self._lst_hours[time_idx]))
        freq = float(self._freq_mhz[freq_idx])
        val = (
            float(self._heatmap_values[time_idx, freq_idx])
            if self._heatmap_values is not None
            else float("nan")
        )
        val_s = f"{val:.3g}" if np.isfinite(val) else "n/a"

        coord = self._coord
        assert coord is not None
        ra_h = coord.ra.to_string(unit=u.hour, precision=1)
        dec_s = coord.dec.to_string(unit=u.deg, precision=1)
        status = (
            f"**{src['name']}** · LST {lst}, {freq:.1f} MHz (t={time_idx}, f={freq_idx}) · "
            f"{self._heatmap_method_label()}={val_s} · RA={ra_h}, Dec={dec_s}"
        )
        if self.heatmap_method == "patch_fit" and self._patch_fit_result is not None:
            status = f"{status}\n\n{_format_patch_fit_diagnostics(self._patch_fit_result, time_idx, freq_idx)}"
        self._set_status(status)
        self._log(
            f"Sky slice — {src['name']}, {self._heatmap_method_label()}={val_s}, "
            f"t={time_idx}, f={freq_idx} ({freq:.1f} MHz)."
        )
        self._update_sky(time_idx, freq_idx)
        _push_panel_layout(self._layout, self._status_pane, self._sky_pane, self._log_pane)

    def _build_heatmap_figure(self, values: np.ndarray):
        n_times, n_freqs = values.shape
        mapper = _color_mapper(values.astype(np.float64, copy=False))
        src = self._current_source
        title_name = src["name"] if src is not None else "LPT"
        method_label = self._heatmap_method_label()
        plot = figure(
            width=1000,
            height=400,
            title=f"{title_name} — {method_label} (click a cell for sky view)",
            x_range=(0, n_times),
            y_range=(0, n_freqs),
            tools="pan,wheel_zoom,reset,tap",
            active_drag="pan",
            active_tap="tap",
        )
        plot.image(
            image=[values.T.astype(np.float64, copy=False)],
            x=0,
            y=0,
            dw=n_times,
            dh=n_freqs,
            color_mapper=mapper,
        )

        time_idx, freq_idx = np.meshgrid(
            np.arange(n_times, dtype=int),
            np.arange(n_freqs, dtype=int),
            indexing="ij",
        )
        flat_time = time_idx.ravel()
        flat_freq = freq_idx.ravel()
        hover_data: dict[str, object] = {
            "x": flat_time + 0.5,
            "y": flat_freq + 0.5,
            "time_idx": flat_time,
            "freq_idx": flat_freq,
            "lst_hour": [
                _format_lst_hour_label(float(h))
                for h in self._lst_hours[flat_time]  # type: ignore[index]
            ],
            "freq_mhz": self._freq_mhz[flat_freq],  # type: ignore[index]
            "value_display": _row_hover(values),
        }
        tooltips: list[tuple[str, str]] = [
            ("LST hour", "@lst_hour"),
            ("Freq (MHz)", "@freq_mhz{0.1}"),
            ("Time idx", "@time_idx"),
            ("Freq idx", "@freq_idx"),
            ("Value", "@value_display"),
        ]
        if self.heatmap_method == "patch_fit" and self._patch_fit_result is not None:
            hover_data.update(_patch_fit_hover_columns(self._patch_fit_result))
            tooltips.extend(
                [
                    ("Patch max (Jy)", "@patch_max_display"),
                    ("χ²_red", "@chi2_display"),
                    ("Fit accepted", "@fit_accepted_display"),
                    ("Peak RA", "@peak_ra_display"),
                    ("Peak Dec", "@peak_dec_display"),
                    ("Offset (l,m px)", "@offset_display"),
                ]
            )

        hover_src = ColumnDataSource(data=hover_data)
        hover_renderer = plot.rect(
            x="x",
            y="y",
            width=1,
            height=1,
            source=hover_src,
            fill_alpha=0,
            line_alpha=0,
        )
        plot.add_tools(HoverTool(renderers=[hover_renderer], tooltips=tooltips))

        def _axis_ticks(
            n: int, axis_values: np.ndarray, fmt: Callable[[float], str]
        ) -> tuple[list[float], dict[float, str]]:
            step = 1 if n <= 24 else int(np.ceil(n / 24))
            indices = range(0, n, step)
            ticks = [i + 0.5 for i in indices]
            labels = {tick: fmt(float(axis_values[i])) for tick, i in zip(ticks, indices, strict=True)}
            return ticks, labels

        x_ticks, x_labels = _axis_ticks(
            n_times, self._lst_hours, _format_lst_hour_label  # type: ignore[arg-type]
        )
        y_ticks, y_labels = _axis_ticks(
            n_freqs, self._freq_mhz, lambda v: f"{float(v):.1f}"  # type: ignore[arg-type]
        )
        plot.xaxis.ticker = FixedTicker(ticks=x_ticks)
        plot.yaxis.ticker = FixedTicker(ticks=y_ticks)
        plot.xaxis.major_label_overrides = x_labels
        plot.yaxis.major_label_overrides = y_labels
        plot.xaxis.axis_label = "LST hour"
        plot.yaxis.axis_label = "Frequency (MHz)"
        plot.xaxis.major_label_orientation = math.pi / 4

        def _on_tap(event: Tap) -> None:
            if event.x is None or event.y is None:
                return
            t_idx = _heatmap_index_from_coord(event.x, n_times)
            f_idx = _heatmap_index_from_coord(event.y, n_freqs)
            _schedule_ipython_main(lambda: self._on_heatmap_tap(t_idx, f_idx))

        plot.on_event(Tap, _on_tap)
        return plot


In [ ]:
from dask.distributed import Client

# Start dask client for parallel processing
client = Client()
print(client)

In [ ]:
review = LPTSourceReview(
    LPT_SOURCES,
    ZARR_PATH,
    patch_scale=PATCH_SCALE,
    sky_fov_deg=SKY_FOV_DEG,
    patch_fit_max_reduced_chi_squared=PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
    heatmap_method=HEATMAP_METHOD,
)
review.panel


## Notes

- **Source list** — LPTs from Rea et al. (2026) with Dec > −20°; maps are computed on demand
  per source and heatmap method (cached for the session).
- **Gray heatmap cells** — NaN or failed values (e.g. patch_fit rejected by χ² cut).
- **Patch size** — `PATCH_SCALE` × max beam FWHM at each time step.
- First extraction on a full cube can take tens of seconds per source depending on Zarr I/O.
